# Self-Supervised Learning, Hands-On
## Rotation Prediction, Contrastive Learning, and Why Representations Collapse

**The core question of SSL:** how do we learn *useful representations* from **unlabeled** data?
Real-world data is mostly unlabeled, and human annotation is expensive. Self-supervised learning
sidesteps this by inventing **"labels for free"** — *pretext tasks* whose answers come directly
from the data itself, forcing an encoder to learn structure that transfers to downstream tasks.

In this notebook we build and train **three runnable demos** on a small image dataset, each
illustrating one SSL family:

| Section | Demo | SSL family | Key idea |
|---|---|---|---|
| 1 | Rotation prediction | **Predictive / pretext** | Predict the applied rotation (0/90/180/270°) — the label is free |
| 2 | SimCLR contrastive | **Contrastive** | Pull two augmented views together, push other images apart (InfoNCE) |
| 3a | Naive positives-only | *(failure mode)* | Drop the negatives → representations **collapse** to a constant |
| 3b | BYOL / SimSiam | **Bootstrapping** | Predictor + stop-gradient + EMA teacher prevents collapse, no negatives |

Everything runs on **Colab CPU or GPU** with deliberately small training budgets — the goal is to
*see* each mechanism work (transfer-accuracy lift, class clustering, std collapse), not to reach
state-of-the-art accuracy. Labels are loaded only for a final *linear-probe evaluation*; they are
**never** used during any SSL training loop.

Run the cells top to bottom. 🚀

In [ ]:
# === C02: Environment setup & reproducibility ===
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """Seed all RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# Device detection (CPU fallback handled by the expression itself)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Optional ipywidgets for the interactive cells (C10, C13) — no hard dependency
try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

print(f"Device:            {device}")
print(f"torch:             {torch.__version__}")
print(f"torchvision:       {torchvision.__version__}")
print(f"numpy:             {np.__version__}")
print(f"scikit-learn:      {sklearn.__version__}")
print(f"ipywidgets ready:  {WIDGETS_AVAILABLE}")

# Sanity checks
assert isinstance(device, torch.device)
seed_everything(SEED); _a = torch.rand(3)
seed_everything(SEED); _b = torch.rand(3)
assert torch.allclose(_a, _b), "Reseeding must reproduce identical draws"
print("Reproducibility check passed:", [round(x, 4) for x in _a.tolist()])

In [ ]:
# === C03: Load a small image dataset (labels used ONLY for downstream eval) ===
import math
import socket
import torchvision.transforms as T

TRAIN_N = 3000   # unlabeled SSL pretraining images
EVAL_N  = 2000   # labeled images, used ONLY for linear-probe evaluation

def _to_chw_float(data: torch.Tensor) -> torch.Tensor:
    """Normalize raw dataset tensor to (N,3,32,32) float in [0,1]."""
    if data.dim() == 3:                       # (N,H,W) grayscale
        data = data.unsqueeze(1).repeat(1, 3, 1, 1)
    elif data.dim() == 4 and data.shape[-1] == 3:  # (N,H,W,3)
        data = data.permute(0, 3, 1, 2)
    data = data.float() / 255.0
    if data.shape[-1] != 32 or data.shape[-2] != 32:
        data = F.interpolate(data, size=(32, 32), mode='bilinear', align_corners=False)
    return data

def _balanced_split(data_t: torch.Tensor, targets: torch.Tensor, train_n: int, eval_n: int):
    """Carve balanced per-class train (unlabeled) and eval (labeled) splits."""
    num_classes = int(targets.max()) + 1
    eval_per_class = eval_n // num_classes
    train_per_class = train_n // num_classes
    g = torch.Generator().manual_seed(SEED)
    train_parts, eval_imgs, eval_lbls = [], [], []
    for c in range(num_classes):
        c_idx = (targets == c).nonzero(as_tuple=True)[0]
        c_idx = c_idx[torch.randperm(c_idx.numel(), generator=g)]
        e_take = min(eval_per_class, c_idx.numel())          # clamp to availability
        eval_c, rest = c_idx[:e_take], c_idx[e_take:]
        t_take = min(train_per_class, rest.numel())
        train_c = rest[:t_take]
        eval_imgs.append(data_t[eval_c]); eval_lbls.append(targets[eval_c])
        train_parts.append(data_t[train_c])
    train_images = torch.cat(train_parts)
    eval_images = torch.cat(eval_imgs)
    eval_labels = torch.cat(eval_lbls)
    # Shuffle so probe train/eval splits (taken as contiguous slices) see all classes
    train_images = train_images[torch.randperm(train_images.shape[0], generator=g)]
    perm = torch.randperm(eval_images.shape[0], generator=g)
    eval_images, eval_labels = eval_images[perm], eval_labels[perm]
    return train_images, eval_images, eval_labels

def make_synthetic_dataset(train_n: int, eval_n: int, num_classes: int = 10):
    """Offline fallback used when no dataset can be downloaded. Each class is a distinct
    oriented colour grating (class- AND rotation-discriminative), so every SSL demo still
    shows real signal (probe lift, clustering, collapse-vs-bootstrapping) without a network."""
    g = torch.Generator().manual_seed(SEED)
    coords = torch.linspace(-1, 1, 32)
    yy, xx = torch.meshgrid(coords, coords, indexing='ij')
    per_class = (train_n + eval_n) // num_classes + 1
    all_imgs, all_lbls = [], []
    for c in range(num_classes):
        ang = math.pi * c / num_classes                    # class-specific orientation
        freq = 2.0 + (c % 5)                                # class-specific spatial frequency
        cx, cy = math.cos(ang), math.sin(ang)
        color = torch.tensor([
            0.5 + 0.5 * math.cos(2 * math.pi * c / num_classes),
            0.5 + 0.5 * math.cos(2 * math.pi * c / num_classes + 2.094),
            0.5 + 0.5 * math.cos(2 * math.pi * c / num_classes + 4.188),
        ])
        phases = torch.rand(per_class, 1, 1, generator=g) * 0.6
        patt = torch.sin(freq * math.pi * (xx * cx + yy * cy) + phases)  # (per_class,32,32)
        patt = (patt + 1.0) / 2.0
        imgs = patt.unsqueeze(1) * color.view(1, 3, 1, 1)              # (per_class,3,32,32)
        imgs = (imgs + 0.05 * torch.randn(per_class, 3, 32, 32, generator=g)).clamp(0, 1)
        all_imgs.append(imgs.float())
        all_lbls.append(torch.full((per_class,), c, dtype=torch.long))
    data_t = torch.cat(all_imgs)
    targets = torch.cat(all_lbls)
    train_images, eval_images, eval_labels = _balanced_split(data_t, targets, train_n, eval_n)
    class_names = [f"class_{i}" for i in range(num_classes)]
    return 'Synthetic', train_images, eval_images, eval_labels, class_names

def load_image_dataset(train_n: int, eval_n: int):
    """Load CIFAR-10 (-> FashionMNIST -> MNIST); fall back to a synthetic class-structured
    dataset if no download is available so the notebook always runs end-to-end."""
    socket.setdefaulttimeout(20)   # fail fast instead of hanging on a stalled download
    dataset, name = None, None
    for loader, dsname in [
        (lambda: torchvision.datasets.CIFAR10(root='./data', train=True, download=True), 'CIFAR10'),
        (lambda: torchvision.datasets.FashionMNIST(root='./data', train=True, download=True), 'FashionMNIST'),
        (lambda: torchvision.datasets.MNIST(root='./data', train=True, download=True), 'MNIST'),
    ]:
        try:
            dataset, name = loader(), dsname
            break
        except Exception as e:
            print(f"{dsname} load failed ({e}); trying fallback...")
    socket.setdefaulttimeout(None)
    if dataset is None:
        print("No image dataset could be downloaded; using a synthetic class-structured dataset.")
        return make_synthetic_dataset(train_n, eval_n)

    data = dataset.data
    data_t = torch.from_numpy(data) if isinstance(data, np.ndarray) else data.clone()
    data_t = _to_chw_float(data_t)

    targets = dataset.targets
    if not torch.is_tensor(targets):
        targets = torch.tensor(np.array(targets))
    targets = targets.long()

    train_images, eval_images, eval_labels = _balanced_split(data_t, targets, train_n, eval_n)
    class_names = list(getattr(dataset, 'classes', [str(i) for i in range(int(targets.max()) + 1)]))
    return name, train_images, eval_images, eval_labels, class_names

DATASET_NAME, train_images, eval_images, eval_labels, class_names = load_image_dataset(TRAIN_N, EVAL_N)

print(f"Dataset: {DATASET_NAME}")
print(f"train_images: {tuple(train_images.shape)}  (SSL pretraining, labels unused)")
print(f"eval_images:  {tuple(eval_images.shape)}   eval_labels: {tuple(eval_labels.shape)}")
print(f"classes ({len(class_names)}): {class_names}")
_uniq, _counts = torch.unique(eval_labels, return_counts=True)
print("eval per-class counts:", _counts.tolist())

# Checks
assert train_images.dtype == torch.float32 and train_images.min() >= 0 and train_images.max() <= 1
assert train_images.shape[1:] == (3, 32, 32)
assert eval_images.shape[0] == eval_labels.shape[0]
assert int(_counts.max() - _counts.min()) <= 1, "eval set should be balanced"

In [ ]:
# === C04: Shared CNN encoder + reusable head factory ===
FEATURE_DIM = 128

class Encoder(nn.Module):
    """Small 3-block CNN backbone, reused (freshly instantiated) by every SSL demo."""
    def __init__(self, feature_dim: int = FEATURE_DIM):
        super().__init__()
        self.feature_dim = feature_dim
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, feature_dim))
        self.pool = nn.AdaptiveAvgPool2d(1)  # any HxW -> fixed vector, avoids shape errors

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.features(x)
        return self.pool(h).flatten(1)       # (B, feature_dim)

def make_head(in_dim: int = FEATURE_DIM, out_dim: int = 10, hidden=None) -> nn.Module:
    """Linear probe / classifier head, or a small MLP when `hidden` is given."""
    if out_dim <= 0:
        raise ValueError("out_dim must be > 0")
    if hidden is None:
        return nn.Linear(in_dim, out_dim)
    return nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, out_dim))

def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

# Checks
_enc = Encoder().to(device)
_out = _enc(torch.randn(4, 3, 32, 32, device=device))
assert _out.shape == (4, FEATURE_DIM)
_head = make_head(out_dim=10)
assert _head(torch.randn(4, FEATURE_DIM)).shape == (4, 10)
assert count_params(Encoder()) > 0
print(f"Encoder feature dim: {FEATURE_DIM}")
print(f"Encoder parameters:  {count_params(Encoder()):,}")

## Section 1 — Predictive pretext tasks: Rotation Prediction

The **predictive** family invents a label that can be computed *automatically* from the input.
Rotation prediction is the cleanest example: take an image, rotate it by one of **0°, 90°, 180°, or
270°**, and ask the encoder to predict *which rotation was applied* — a 4-class classification problem.

**Why does this teach anything useful?** To answer "which way is up?" the network can't rely on
low-level texture alone — it must recognize **object structure** (a cat's legs point down, the sky
is up, text reads left-to-right). The supervision is *free*: we generated the rotation labels
ourselves, no human annotation required.

We will train the encoder on this task, then **freeze it** and check whether the features it learned
transfer to real classification via a linear probe.

In [ ]:
# === C06: Build the rotation pretext data (labels for free) ===
NUM_ROTATIONS = 4

def rotate_image(x: torch.Tensor, k: int) -> torch.Tensor:
    """Rotate a (C,H,W) or (B,C,H,W) tensor by k*90 degrees."""
    if x.dim() == 3:
        x = x.unsqueeze(0)
    assert x.dim() >= 3, "expected at least 3 dims"
    return torch.rot90(x, k % NUM_ROTATIONS, dims=(-2, -1))

def make_rotation_batch(images: torch.Tensor, idx=None):
    """Return (4B images, 4B auto rotation labels in {0,1,2,3}) from unlabeled images."""
    if images.dim() == 3:
        images = images.unsqueeze(0)
    if idx is not None:
        images = images[idx]
    views, labels = [], []
    for k in range(NUM_ROTATIONS):
        views.append(torch.rot90(images, k, dims=(-2, -1)))
        labels.append(torch.full((images.shape[0],), k, dtype=torch.long))
    return torch.cat(views, dim=0), torch.cat(labels, dim=0)

# Visualize the free supervision signal: two images x four rotations
try:
    deg = [0, 90, 180, 270]
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for r in range(2):
        for k in range(NUM_ROTATIONS):
            rot = rotate_image(train_images[r], k)[0].permute(1, 2, 0).cpu().numpy()
            axes[r, k].imshow(np.clip(rot, 0, 1))
            axes[r, k].set_title(f"label={k}  ({deg[k]}°)", fontsize=10)
            axes[r, k].axis('off')
    plt.suptitle("Rotation pretext task — the label IS the rotation we applied (free supervision)")
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Display skipped: {e}")

# Checks
_imgs, _lbls = make_rotation_batch(train_images[:8])
assert _imgs.shape[0] == 8 * 4 and _lbls.shape[0] == 8 * 4
assert set(_lbls.tolist()) <= {0, 1, 2, 3}
_x = train_images[0]
assert torch.allclose(rotate_image(rotate_image(_x, 1), 3), rotate_image(_x, 0)), "rot90 then rot270 == identity"
print("Rotation batch OK:", tuple(_imgs.shape), "labels:", sorted(set(_lbls.tolist())))

In [ ]:
# === C07: Train the encoder on the 4-way rotation task ===
EPOCHS_ROT = 5
BATCH_SIZE = 128
STEPS_PER_EPOCH = 20

def train_rotation(encoder, head, images, epochs=EPOCHS_ROT, batch_size=BATCH_SIZE, lr=1e-3):
    opt = torch.optim.Adam(list(encoder.parameters()) + list(head.parameters()), lr=lr)
    history = {'loss': [], 'acc': []}
    n = images.shape[0]
    for epoch in range(epochs):
        encoder.train(); head.train()
        ep_loss, ep_correct, ep_total = 0.0, 0, 0
        for _ in range(STEPS_PER_EPOCH):
            sel = torch.randint(0, n, (batch_size,))
            rot_imgs, rot_lbls = make_rotation_batch(images[sel])
            try:
                rot_imgs, rot_lbls = rot_imgs.to(device), rot_lbls.to(device)
                logits = head(encoder(rot_imgs))
                loss = F.cross_entropy(logits, rot_lbls)
                opt.zero_grad(); loss.backward(); opt.step()
            except RuntimeError as e:
                if 'out of memory' in str(e).lower() and batch_size > 16:
                    torch.cuda.empty_cache()
                    print("CUDA OOM — retrying this step with a smaller batch.")
                    batch_size = batch_size // 2
                    continue
                raise
            ep_loss += loss.item() * rot_lbls.size(0)
            ep_correct += (logits.argmax(1) == rot_lbls).sum().item()
            ep_total += rot_lbls.size(0)
        history['loss'].append(ep_loss / ep_total)
        history['acc'].append(ep_correct / ep_total)
        print(f"[rotation] epoch {epoch+1}/{epochs}  loss={history['loss'][-1]:.4f}  acc={history['acc'][-1]:.3f}")
    return history

seed_everything(SEED)
rotation_encoder = Encoder().to(device)
rotation_head = make_head(out_dim=NUM_ROTATIONS).to(device)
rotation_train_history = train_rotation(rotation_encoder, rotation_head, train_images)

assert len(rotation_train_history['loss']) == EPOCHS_ROT
assert next(rotation_encoder.parameters()).device.type == device.type
print("Rotation accuracy improved over training:",
      rotation_train_history['acc'][-1] > rotation_train_history['acc'][0])

In [ ]:
# === C08: Linear-probe evaluation (the payoff of free supervision) ===
def extract_features(encoder, images, batch_size: int = 256):
    """Frozen forward pass -> CPU float32 feature matrix (N, FEATURE_DIM)."""
    encoder.eval()
    feats = []
    with torch.no_grad():
        for i in range(0, images.shape[0], batch_size):
            f = encoder(images[i:i + batch_size].to(device)).cpu().numpy()
            feats.append(f)
    out = np.concatenate(feats, axis=0).astype(np.float32)
    return out.reshape(out.shape[0], -1)

PROBE_SPLIT_IDX = None

def linear_probe_accuracy(features, labels, train_frac: float = 0.7) -> float:
    """Fit a logistic-regression probe on frozen features; return test accuracy."""
    global PROBE_SPLIT_IDX
    n = features.shape[0]
    split = int(n * train_frac)
    PROBE_SPLIT_IDX = split  # fixed boundary reused by C13/C17 for a fair comparison
    clf = LogisticRegression(max_iter=1000)
    clf.fit(features[:split], labels[:split])
    return float(clf.score(features[split:], labels[split:]))

eval_labels_np = eval_labels.numpy()

seed_everything(SEED)
random_encoder = Encoder().to(device)  # untrained baseline

eval_features_rotation = extract_features(rotation_encoder, eval_images)
eval_features_random = extract_features(random_encoder, eval_images)
rotation_probe_acc = linear_probe_accuracy(eval_features_rotation, eval_labels_np)
random_probe_acc = linear_probe_accuracy(eval_features_random, eval_labels_np)

print(f"Random-encoder    linear-probe acc: {random_probe_acc:.3f}")
print(f"Rotation-pretrain linear-probe acc: {rotation_probe_acc:.3f}")
print("(expected: rotation >= random — features learned with NO human labels transfer)")

try:
    plt.figure(figsize=(4, 4))
    plt.bar(['random', 'rotation'], [random_probe_acc, rotation_probe_acc],
            color=['gray', 'steelblue'])
    plt.ylabel('linear-probe accuracy'); plt.ylim(0, 1)
    plt.title('Downstream accuracy: random vs rotation-pretrained')
    plt.show()
except Exception as e:
    print(e)

# Checks
assert 0.0 <= rotation_probe_acc <= 1.0 and 0.0 <= random_probe_acc <= 1.0
assert eval_features_rotation.shape[0] == eval_labels.shape[0]

## Section 2 — Contrastive Learning & Data Augmentation

Rotation prediction is clever but indirect. **Contrastive learning** attacks representation learning
head-on with a simple principle:

> Two augmented views of the **same** image should map to **nearby** embeddings (a *positive pair*);
> views of **different** images should map **far apart** (*negatives*).

**Data augmentation is the heart of the method.** By aggressively cropping, flipping, jittering color,
and blurring, we create two views that look different at the pixel level but share the same semantic
content. Forcing them together teaches the encoder to be *invariant* to those nuisances and to keep
only what identifies the object.

The **InfoNCE / NT-Xent** loss formalizes this: for each view, treat its partner as the one positive
among a batch of negatives, and minimize a softmax cross-entropy over cosine similarities scaled by a
temperature τ. We then visualize that same-class images cluster — **without the model ever seeing a
label**.

In [ ]:
# === C10: SimCLR augmentation pipeline + interactive two-views panel ===
import torchvision.transforms as T

def make_ssl_augment(crop_scale: float = 0.6, jitter_strength: float = 0.5, blur: bool = True):
    """Build a configurable SimCLR-style augmentation operating on (3,H,W) float tensors."""
    crop_scale = float(min(max(crop_scale, 0.05), 1.0))   # clamp to (0,1]
    s = float(max(jitter_strength, 0.0))                  # strength >= 0
    tfs = [
        T.RandomResizedCrop(32, scale=(crop_scale, 1.0), antialias=True),
        T.RandomHorizontalFlip(),
        T.RandomApply([T.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)], p=0.8),
        T.RandomGrayscale(p=0.2),
    ]
    if blur:
        tfs.append(T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 2.0))], p=0.5))
    return T.Compose(tfs)

ssl_augment = make_ssl_augment()

def two_views(images: torch.Tensor, augment=None):
    """Return two independently augmented tensors (positive pairs) of a batch."""
    aug = augment or ssl_augment
    if images.dim() == 3:
        images = images.unsqueeze(0)
    v1 = torch.stack([aug(img) for img in images]).clamp(0, 1)
    v2 = torch.stack([aug(img) for img in images]).clamp(0, 1)
    return v1, v2

def _show_two_views(image: torch.Tensor, augment) -> None:
    v1, v2 = two_views(image.unsqueeze(0), augment)
    fig, ax = plt.subplots(1, 3, figsize=(9, 3))
    for a, im, t in zip(ax,
                        [image, v1[0], v2[0]],
                        ['original', 'view 1', 'view 2']):
        a.imshow(np.clip(im.permute(1, 2, 0).cpu().numpy(), 0, 1)); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.show()

_sample_img = train_images[0]
if WIDGETS_AVAILABLE:
    def _update(crop_scale=0.6, jitter_strength=0.5, blur=True):
        _show_two_views(_sample_img, make_ssl_augment(crop_scale, jitter_strength, blur))
    widgets.interact(
        _update,
        crop_scale=widgets.FloatSlider(min=0.1, max=1.0, step=0.1, value=0.6, description='crop scale'),
        jitter_strength=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.5, description='jitter'),
        blur=widgets.Checkbox(value=True, description='blur'),
    )
else:
    print("ipywidgets unavailable — showing a static two-view figure.")
    _show_two_views(_sample_img, ssl_augment)

# Checks
_v1, _v2 = two_views(train_images[:4])
assert _v1.shape == _v2.shape == (4, 3, 32, 32)
assert not torch.allclose(_v1, _v2)          # random augmentation makes the views differ
assert _v1.min() >= 0 and _v1.max() <= 1

In [ ]:
# === C11: InfoNCE / NT-Xent loss + projection head ===
class ProjectionHead(nn.Module):
    """MLP projection head (FEATURE_DIM -> hidden -> out_dim), used during contrastive training."""
    def __init__(self, in_dim: int = FEATURE_DIM, hidden: int = 128, out_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class ContrastiveModel(nn.Module):
    """Encoder + projection head; forward returns L2-normalized projected embeddings."""
    def __init__(self, encoder, proj_dim: int = 64):
        super().__init__()
        self.encoder = encoder
        self.projector = ProjectionHead(FEATURE_DIM, 128, proj_dim)
    def forward(self, x):
        return F.normalize(self.projector(self.encoder(x)), dim=1)

TEMPERATURE = 0.5

def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = TEMPERATURE):
    """NT-Xent loss: each view's partner is its single positive among 2B-2 negatives."""
    if z1.shape != z2.shape:
        raise ValueError("z1 and z2 must have the same shape (B, D)")
    assert temperature > 0, "temperature must be > 0"
    B = z1.shape[0]
    z = torch.cat([F.normalize(z1, dim=1), F.normalize(z2, dim=1)], dim=0)  # (2B, D)
    sim = (z @ z.t()) / temperature                                         # (2B, 2B)
    sim.masked_fill_(torch.eye(2 * B, dtype=torch.bool, device=z.device), float('-inf'))
    targets = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, targets)

# Sanity checks
_z = F.normalize(torch.randn(8, 64), dim=1)
_aligned = info_nce_loss(_z, _z.clone()).item()
_random = info_nce_loss(_z, F.normalize(torch.randn(8, 64), dim=1)).item()
print(f"InfoNCE  aligned={_aligned:.4f}  random={_random:.4f}")
assert _aligned < _random and np.isfinite(_aligned)
_model = ContrastiveModel(Encoder())
_o = _model(torch.randn(4, 3, 32, 32))
assert _o.shape[0] == 4 and torch.allclose(_o.norm(dim=1), torch.ones(4), atol=1e-4)
print("InfoNCE / ContrastiveModel sanity checks passed.")

In [ ]:
# === C12: Train the contrastive (SimCLR-style) model on positive pairs ===
EPOCHS_CON = 8
CON_STEPS_PER_EPOCH = 20

def train_contrastive(model, images, epochs=EPOCHS_CON, batch_size=128, lr=1e-3, temperature=TEMPERATURE):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'loss': []}
    n = images.shape[0]
    for epoch in range(epochs):
        model.train()
        ep_loss, count = 0.0, 0
        for _ in range(CON_STEPS_PER_EPOCH):
            sel = torch.randint(0, n, (batch_size,))
            batch = images[sel]
            if batch.shape[0] < 2:           # contrastive loss needs >= 2 examples
                continue
            v1, v2 = two_views(batch)
            try:
                v1, v2 = v1.to(device), v2.to(device)
                z1, z2 = model(v1), model(v2)
                loss = info_nce_loss(z1, z2, temperature)
                opt.zero_grad(); loss.backward(); opt.step()
            except RuntimeError as e:
                if 'out of memory' in str(e).lower() and batch_size > 16:
                    torch.cuda.empty_cache(); batch_size //= 2
                    print("CUDA OOM — halving batch and retrying.")
                    continue
                raise
            ep_loss += loss.item(); count += 1
        history['loss'].append(ep_loss / max(count, 1))
        print(f"[contrastive] epoch {epoch+1}/{epochs}  InfoNCE={history['loss'][-1]:.4f}")
    return history

seed_everything(SEED)
contrastive_model = ContrastiveModel(Encoder()).to(device)
contrastive_train_history = train_contrastive(contrastive_model, train_images)
contrastive_encoder = contrastive_model.encoder  # discard projection head (standard SimCLR practice)

assert len(contrastive_train_history['loss']) == EPOCHS_CON
assert isinstance(contrastive_encoder, Encoder)
print("Contrastive loss did not increase:",
      contrastive_train_history['loss'][-1] <= contrastive_train_history['loss'][0])

In [ ]:
# === C13: Visualize the learned embedding space (same-class clustering) ===
def reduce_2d(features, method: str = 'tsne'):
    """Reduce features to 2D via PCA-then-TSNE, falling back to PCA(2) on error."""
    if method == 'tsne':
        try:
            x = features
            if x.shape[1] > 50:
                x = PCA(n_components=50, random_state=SEED).fit_transform(x)
            perplexity = min(30, max(5, features.shape[0] // 4))
            emb = TSNE(n_components=2, init='pca', perplexity=perplexity,
                       random_state=SEED).fit_transform(x)
            return emb, 'tsne'
        except Exception as e:
            print(f"TSNE failed ({e}); falling back to PCA(2).")
    return PCA(n_components=2, random_state=SEED).fit_transform(features), 'pca'

def _scatter_embedding(ax, emb2d, labels, class_names, title):
    for c in range(len(class_names)):
        m = labels == c
        if m.any():
            ax.scatter(emb2d[m, 0], emb2d[m, 1], s=8, alpha=0.6, label=class_names[c])
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

# Subsample for TSNE speed
MAX_PTS = 800
seed_everything(SEED)
sub = np.random.permutation(eval_images.shape[0])[:MAX_PTS]
sub_labels = eval_labels.numpy()[sub]
print(f"Embedding subsample size: {len(sub)}")

seed_everything(SEED)
random_encoder_viz = Encoder().to(device)
feat_con = extract_features(contrastive_encoder, eval_images[sub])
feat_rand = extract_features(random_encoder_viz, eval_images[sub])
contrastive_embedding_2d, _mc = reduce_2d(feat_con)
random_embedding_2d, _mr = reduce_2d(feat_rand)
print(f"2D reduction used: contrastive={_mc}, random={_mr}")

def _draw(which='contrastive'):
    fig, ax = plt.subplots(figsize=(6, 5))
    if which == 'contrastive':
        _scatter_embedding(ax, contrastive_embedding_2d, sub_labels, class_names, 'Contrastive encoder')
    else:
        _scatter_embedding(ax, random_embedding_2d, sub_labels, class_names, 'Random encoder')
    ax.legend(fontsize=6, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.show()

if WIDGETS_AVAILABLE:
    widgets.interact(_draw, which=widgets.Dropdown(options=['contrastive', 'random'],
                                                   value='contrastive', description='encoder'))
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    _scatter_embedding(axes[0], random_embedding_2d, sub_labels, class_names, 'Random encoder')
    _scatter_embedding(axes[1], contrastive_embedding_2d, sub_labels, class_names, 'Contrastive encoder')
    axes[1].legend(fontsize=6, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.show()

# Checks
assert contrastive_embedding_2d.shape[1] == 2
assert contrastive_embedding_2d.shape[0] == random_embedding_2d.shape[0]

## Section 3 — Representation Collapse & Bootstrapping

Contrastive learning needs **negatives** to work — they provide the "push apart" force. This raises a
tempting question:

> What if we drop the negatives entirely and *only* pull positive pairs together?

There is a trivial way to make every positive pair identical: map **every** image to the **same
constant vector**. The loss becomes zero, but the representation is useless. This failure is called
**representation collapse**, and we will reproduce it: the loss drops to ~0 while the embedding
standard deviation (our collapse indicator) crashes toward 0.

**The fix — bootstrapping (BYOL / SimSiam).** Surprisingly, you *can* learn from positives only,
without negatives, using three ingredients:

1. an extra **predictor** MLP on the online network,
2. a **stop-gradient** on the target branch, and
3. a **target encoder** that is an **EMA** (exponential moving average) of the online encoder.

The asymmetry breaks the collapse solution. Below we train both and plot their embedding-std
trajectories side by side.

In [ ]:
# === C15: Demonstrate collapse (naive positives-only objective) ===
def embedding_std(embeddings: torch.Tensor) -> float:
    """Mean per-dimension std of L2-normalized embeddings — the collapse indicator."""
    z = F.normalize(embeddings, dim=1, eps=1e-6)
    return z.std(dim=0).mean().item()

COLLAPSE_STEPS = 100

def train_collapse(encoder, images, steps=COLLAPSE_STEPS, batch_size=128, lr=1e-3):
    """Pull two views together with NO negatives and NO stop-gradient -> collapse."""
    proj = ProjectionHead(FEATURE_DIM, 128, 64).to(device)
    opt = torch.optim.Adam(list(encoder.parameters()) + list(proj.parameters()), lr=lr)
    history = {'step': [], 'loss': [], 'std': []}
    n = images.shape[0]
    encoder.train(); proj.train()
    for step in range(steps):
        sel = torch.randint(0, n, (batch_size,))
        v1, v2 = two_views(images[sel])
        v1, v2 = v1.to(device), v2.to(device)
        z1 = F.normalize(proj(encoder(v1)), dim=1)
        z2 = F.normalize(proj(encoder(v2)), dim=1)
        loss = ((z1 - z2) ** 2).sum(dim=1).mean()    # symmetric MSE, no negatives/stop-grad
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():                        # std on encoder features, comparable to C16
            std = embedding_std(encoder(torch.cat([v1, v2], dim=0)))
        history['step'].append(step); history['loss'].append(loss.item()); history['std'].append(std)
        if step % 20 == 0:
            print(f"[collapse] step {step:3d}  loss={loss.item():.4f}  std={std:.4f}")
    return history

seed_everything(SEED)
collapse_encoder = Encoder().to(device)
collapse_history = train_collapse(collapse_encoder, train_images)

try:
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(collapse_history['step'], collapse_history['loss'], 'r-', label='loss')
    ax1.set_xlabel('step'); ax1.set_ylabel('loss', color='r'); ax1.tick_params(axis='y', labelcolor='r')
    ax2 = ax1.twinx()
    ax2.plot(collapse_history['step'], collapse_history['std'], 'b-', label='embedding std')
    ax2.set_ylabel('embedding std', color='b'); ax2.tick_params(axis='y', labelcolor='b')
    plt.title('Naive positives-only: loss -> 0 while embedding std collapses -> 0')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(e)

# Checks
assert len(collapse_history['std']) == len(collapse_history['loss'])
print("embedding std decreased (collapse):", collapse_history['std'][-1] < collapse_history['std'][0])

In [ ]:
# === C16: Bootstrapping fix (SimSiam/BYOL: predictor + stop-grad + EMA target) ===
import copy

EMA_MOMENTUM = 0.99
BYOL_STEPS = COLLAPSE_STEPS  # same budget as the collapse demo for a fair comparison

def ema_update(target: nn.Module, online: nn.Module, momentum: float = EMA_MOMENTUM) -> None:
    """target = m * target + (1 - m) * online, under no_grad."""
    with torch.no_grad():
        for tp, op in zip(target.parameters(), online.parameters()):
            tp.data.mul_(momentum).add_(op.data, alpha=1 - momentum)

def negative_cosine(p: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """Negative cosine similarity with a stop-gradient on the target z."""
    p = F.normalize(p, dim=1)
    z = F.normalize(z.detach(), dim=1)        # stop-gradient
    return -(p * z).sum(dim=1).mean()

def train_bootstrap(images, steps=BYOL_STEPS, batch_size=128, lr=1e-3, momentum=EMA_MOMENTUM):
    assert 0 <= momentum < 1, "momentum must be in [0, 1)"
    online_enc = Encoder().to(device)
    online_proj = ProjectionHead(FEATURE_DIM, 128, 64).to(device)
    predictor = make_head(64, 64, hidden=128).to(device)
    target_enc = copy.deepcopy(online_enc)
    target_proj = copy.deepcopy(online_proj)
    for p in list(target_enc.parameters()) + list(target_proj.parameters()):
        p.requires_grad_(False)               # target is never optimized directly

    params = list(online_enc.parameters()) + list(online_proj.parameters()) + list(predictor.parameters())
    opt = torch.optim.Adam(params, lr=lr)
    history = {'step': [], 'loss': [], 'std': []}
    n = images.shape[0]
    for step in range(steps):
        online_enc.train(); online_proj.train(); predictor.train()
        sel = torch.randint(0, n, (batch_size,))
        v1, v2 = two_views(images[sel])
        v1, v2 = v1.to(device), v2.to(device)
        p1 = predictor(online_proj(online_enc(v1)))
        p2 = predictor(online_proj(online_enc(v2)))
        with torch.no_grad():
            t1 = target_proj(target_enc(v1))
            t2 = target_proj(target_enc(v2))
        loss = 0.5 * (negative_cosine(p1, t2) + negative_cosine(p2, t1))
        opt.zero_grad(); loss.backward(); opt.step()
        ema_update(target_enc, online_enc, momentum)
        ema_update(target_proj, online_proj, momentum)
        with torch.no_grad():
            std = embedding_std(online_enc(torch.cat([v1, v2], dim=0)))
        history['step'].append(step); history['loss'].append(loss.item()); history['std'].append(std)
        if step % 20 == 0:
            print(f"[bootstrap] step {step:3d}  loss={loss.item():.4f}  std={std:.4f}")
    return history, online_enc

seed_everything(SEED)
byol_history, bootstrap_encoder = train_bootstrap(train_images)

try:
    plt.figure(figsize=(7, 4))
    plt.plot(collapse_history['step'], collapse_history['std'], 'r-', label='naive (collapse)')
    plt.plot(byol_history['step'], byol_history['std'], 'g-', label='bootstrapping (BYOL/SimSiam)')
    plt.xlabel('step'); plt.ylabel('embedding std'); plt.legend()
    plt.title('Embedding std: bootstrapping avoids collapse without any negatives')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(e)

# Checks
assert len(byol_history['std']) == len(byol_history['loss'])
print("bootstrapping keeps std above the collapsed baseline:",
      byol_history['std'][-1] > collapse_history['std'][-1])

In [ ]:
# === C17: Side-by-side summary across all four encoders ===
def summarize(eval_images, eval_labels):
    labels_np = eval_labels.numpy()
    con_acc = linear_probe_accuracy(extract_features(contrastive_encoder, eval_images), labels_np)
    boot_acc = linear_probe_accuracy(extract_features(bootstrap_encoder, eval_images), labels_np)
    probe_acc = {
        'random': random_probe_acc,
        'rotation': rotation_probe_acc,
        'contrastive': con_acc,
        'bootstrapping': boot_acc,
    }
    return probe_acc, con_acc, boot_acc

probe_acc, contrastive_probe_acc, bootstrap_probe_acc = summarize(eval_images, eval_labels)
summary_comparison = {
    'probe_acc': probe_acc,
    'std_curves': {'naive': collapse_history['std'], 'bootstrapping': byol_history['std']},
}

print("Downstream linear-probe accuracy by SSL method")
print("-" * 44)
for k, v in probe_acc.items():
    print(f"  {k:14s}: {v:.3f}")

try:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(collapse_history['step'], collapse_history['std'], 'r-', label='naive (collapse)')
    axes[0].plot(byol_history['step'], byol_history['std'], 'g-', label='bootstrapping')
    axes[0].set_xlabel('step'); axes[0].set_ylabel('embedding std')
    axes[0].set_title('Collapse vs bootstrapping'); axes[0].legend()
    methods = list(probe_acc.keys()); vals = [probe_acc[m] for m in methods]
    axes[1].bar(methods, vals, color=['gray', 'steelblue', 'seagreen', 'darkorange'])
    axes[1].set_ylabel('linear-probe accuracy'); axes[1].set_ylim(0, 1)
    axes[1].set_title('Downstream accuracy by SSL method')
    for i, v in enumerate(vals):
        axes[1].text(i, v + 0.01, f"{v:.2f}", ha='center')
    plt.setp(axes[1].get_xticklabels(), rotation=20)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(e)

# Checks
assert set(summary_comparison['probe_acc'].keys()) == {'random', 'rotation', 'contrastive', 'bootstrapping'}
assert all(0.0 <= v <= 1.0 for v in summary_comparison['probe_acc'].values())

## Wrap-up — The Five SSL Families

We built three SSL mechanisms from scratch and saw each one work:

- **Predictive** (rotation) — invent a free label; solving it forces structural understanding.
- **Contrastive** (SimCLR / InfoNCE) — positives together, negatives apart; same-class images cluster.
- **Bootstrapping** (BYOL / SimSiam) — predictor + stop-gradient + EMA teacher learns from positives
  *only*, dodging the collapse that sinks a naive positives-only objective.

These sit inside the lecture's broader **5-family taxonomy**, which spans both image and speech:

| Family | Image example | Speech example |
|---|---|---|
| Generative / reconstructive | MAE, iGPT | Mockingjay, APC |
| Predictive (pretext) | Rotation, Jigsaw | — |
| Contrastive | SimCLR, MoCo | Wav2vec 2.0, CPC |
| Bootstrapping | BYOL, SimSiam | HuBERT, Data2vec |
| Regularization (redundancy reduction) | Barlow Twins, VICReg | DeLoRes |

*(We implemented the predictive, contrastive, and bootstrapping rows; the generative and
regularization rows are covered here as narrative.)*

### Try it yourself 🧪
- **Stronger / weaker augmentations** — push the C10 sliders and re-run contrastive training. How
  does the embedding clustering in C13 change?
- **Longer training** — raise `EPOCHS_CON` / `COLLAPSE_STEPS` and watch the std curves and probe
  accuracies move.
- **MoCo-style negative queue** — extend C11 with a memory bank of past embeddings as extra negatives.
- **Predictor ablation** — remove the predictor in C16 and confirm bootstrapping then collapses too.

The takeaway: with the right pretext, objective, or architectural asymmetry, an encoder can learn
genuinely transferable representations from data that was never labeled.